
# GGMP Synthetic Non-Gaussian Benchmark

This notebook builds a synthetic distribution-valued dataset from a **single latent function** `f(x)`.
The output density transitions smoothly between regimes with strong overlap (effectively one bulk) and regimes with clearly separated multimodality.

It then runs the current GGMP pipeline (`ggmp.py`)


In [ ]:

from __future__ import annotations

import os
import time
import logging
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

import importlib
import ggmp as ggmp_mod
importlib.reload(ggmp_mod)

from ggmp import (
    GGMP,
    hyperparameters,
    _get_key,
    gaussian_pdf,
    empirical_pdf_from_samples,
    fit_gmm_fixed_weights,
    build_gp_init_kwargs,
    _parse_device_ids,
    _normalize_pdf,
    train_component_gps_mcmc,
    prepare_station_terms_density,
    optimize_weights_em_density,
    bhattacharyya_distance,
    kl_divergence,
    wasserstein_1d,
)


In [ ]:

# --------------------------
# Toggle-friendly parameters
# --------------------------
FAST_MODE = False
SEED = 42

# Synthetic dataset size
N_STATIONS = 120 if FAST_MODE else 1000
SAMPLES_PER_STATION = 800 if FAST_MODE else 5000
TRAIN_RATIO = 0.8

# Domain used to build synthetic station densities
Y_DOMAIN = np.linspace(-8.0, 8.0, 1200)
X_MIN, X_MAX = -3.0, 3.0

# Sweep settings
K_SWEEP = [1, 3, 5, 10]
HIST_BINS = 140

# Station-level fixed-weight GMM fit
GMM_MAX_ITER = 120
GMM_TOL = 1e-4

# GP MCMC settings
MCMC_UNTIL_CONVERGED = True
N_UPDATES_GP = 140 if FAST_MODE else 500
MCMC_CHUNK = 50 if FAST_MODE else 100
MCMC_MAX_TOTAL = 600 if FAST_MODE else 5000
MCMC_TOL_REL = 5e-3 if FAST_MODE else 1e-3
MCMC_PATIENCE = 2 if FAST_MODE else 5

GP_PARALLEL = True
GP_WORKERS = None
BLAS_THREADS_PER_GP = 1

# Save traces/artifacts
SAVE_GP_MCMC = True
SAVE_GP_MCMC_CHUNKS = True
GP_MCMC_THIN = 1
OUT_DIR = Path("ggmp_em_runs") / "synthetic_disjoint_modes"

# EM weight fit
WEIGHT_FLOOR = 1e-6
EM_MAX_ITER = 140
EM_TOL_L1 = 1e-10
EM_LOG_EVERY = 10

# Plotting
N_PLOT_STATIONS = 8
N_COMPARISON_STATIONS = 4
PLOT_BINS = 35
FIG_DPI = 170

# Compute threading
N_THREADS = 1

# GPU options
USE_GPU = False
GPU_ENGINE = "torch"
GP_DEVICE_IDS = None

# --------------------------
# Repro + logging
# --------------------------
np.random.seed(SEED)
os.environ["NTHREADS"] = str(int(N_THREADS))
os.environ["OPENBLAS_NUM_THREADS"] = str(int(N_THREADS))
os.environ["MKL_NUM_THREADS"] = str(int(N_THREADS))
os.environ["OMP_NUM_THREADS"] = str(int(N_THREADS))

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Publication-friendly defaults
plt.rcParams.update({
    "figure.dpi": FIG_DPI,
    "savefig.dpi": FIG_DPI,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "legend.fontsize": 9,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

OUT_DIR.mkdir(parents=True, exist_ok=True)
logging.info("Output root: %s", OUT_DIR)



## Synthetic data construction

A single nonlinear mean function `f(x)` controls the density location.
A smooth separation scale `s(x)` controls whether modes are heavily overlapping or widely separated.
Internal subcomponent mixtures are asymmetric and heteroscedastic to make local shapes clearly non-Gaussian.


In [ ]:


def latent_mean(x: float) -> float:
    """Single nonlinear latent function driving all density structure."""
    x = float(x)
    return float(
        0.95 * np.sin(1.05 * x - 0.30)
        + 0.55 * np.sin(2.45 * x + 0.80)
        - 0.38 * np.tanh(1.8 * x)
        + 0.075 * x**3
    )


def _sigmoid(z: np.ndarray | float) -> np.ndarray | float:
    return 1.0 / (1.0 + np.exp(-np.asarray(z)))


def separation_scale(x: float) -> float:
    """
    Smooth split strength: near 0 => almost one bulk, large => strongly multimodal.
    """
    x = float(x)
    z = (
        1.35 * np.sin(1.15 * x - 0.20)
        - 1.05 * np.cos(2.35 * x + 0.55)
        + 0.60 * np.sin(3.80 * x - 0.65)
    )
    split = float(_sigmoid(1.45 * z))
    return float(0.03 + 3.35 * split**1.25)


def _softmax(z: np.ndarray) -> np.ndarray:
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / np.sum(e)


def mode_center_offsets(x: float, sep: float) -> np.ndarray:
    """Three centers tied to the same f(x), with nonlinear split and bend."""
    x = float(x)
    bend = 0.55 * np.sin(1.60 * x + 0.20) + 0.35 * np.sin(3.10 * x - 0.70)

    left = -sep * (0.95 + 0.25 * np.sin(0.90 * x - 0.40)) + 0.22 * bend
    mid = 0.30 * sep * np.sin(1.35 * x + 0.50)
    right = sep * (0.82 + 0.30 * np.cos(0.80 * x + 0.60)) - 0.22 * bend

    return np.array([left, mid, right], dtype=float)


def mode_weights(x: float, sep: float) -> np.ndarray:
    """
    Smoothly switches between near-single-bulk and 2/3-active-mode regimes.
    """
    x = float(x)
    split_gate = float(_sigmoid((sep - 1.00) / 0.22))

    left_pref = float(_sigmoid(3.8 * (np.sin(1.45 * x - 0.35) - 0.15)))
    right_pref = float(_sigmoid(3.6 * (np.cos(1.20 * x + 0.75) - 0.10)))

    center_strength = 0.30 + 2.85 * (1.0 - split_gate) + 0.35 * float(_sigmoid(2.0 * np.sin(0.70 * x)))
    outer_strength = split_gate**1.10

    w = np.array(
        [
            0.015 + 1.75 * outer_strength * left_pref,
            center_strength,
            0.015 + 1.75 * outer_strength * right_pref,
        ],
        dtype=float,
    )
    w = np.maximum(w, 1e-12)
    return w / np.sum(w)


SUB_OFFSETS = np.array([-1.60, -0.35, 0.55, 1.55], dtype=float)


def mode_inner_weights(x: float, mode_id: int) -> np.ndarray:
    """Strongly varying asymmetric subcomponent weights (non-Gaussian per mode)."""
    x = float(x)
    z = np.array(
        [
            2.1 * np.sin((0.95 + 0.10 * mode_id) * x + 0.45 * mode_id),
            2.6 * np.cos((1.18 + 0.08 * mode_id) * x - 0.20),
            1.8 * np.sin((1.60 + 0.03 * mode_id) * x + 1.05),
            2.2 * np.cos((0.88 + 0.06 * mode_id) * x + 0.70),
        ],
        dtype=float,
    )
    w = _softmax(z)
    w = w**2.25
    return w / np.sum(w)


def build_station_density(x: float, domain: np.ndarray) -> tuple[np.ndarray, dict]:
    """Non-Gaussian station density from one latent function with overlap/split regimes."""
    x = float(x)
    f = latent_mean(x)
    sep = separation_scale(x)

    c_off = mode_center_offsets(x, sep)
    centers = f + c_off
    w_mode = mode_weights(x, sep)

    density = np.zeros_like(domain, dtype=float)
    mode_meta = []

    # high when modes are mostly merged; low when strongly split
    overlap_factor = float(_sigmoid((1.20 - sep) / 0.22))

    for b, center in enumerate(centers):
        w_in = mode_inner_weights(x, b)
        comp_meta = []

        for j, off in enumerate(SUB_OFFSETS):
            # shrink internal offsets in strong-overlap zones, expand in split zones
            local_scale = 0.06 + 0.34 * (1.0 - overlap_factor) + 0.07 * np.abs(np.sin(1.9 * x + 0.6 * (b + j)))
            local_offset = off * float(local_scale)

            # additional smooth skew/wiggle to avoid near-Gaussian shapes
            skew_bend = 0.18 * np.sin(2.4 * x + 0.55 * b + 0.85 * j) * (1.0 - overlap_factor)
            mu = float(center + local_offset + skew_bend)

            base_sigma = 0.05 + 0.11 * overlap_factor + 0.05 * (0.5 + 0.5 * np.sin(1.7 * x + 0.35 * (b + 2 * j)))
            edge_boost = 0.10 * (1.0 - overlap_factor) * (1.0 if j in (0, len(SUB_OFFSETS) - 1) else 0.45)
            sigma = float(max(base_sigma + edge_boost, 0.045))
            var = sigma**2

            w_comp = float(w_mode[b] * w_in[j])
            density += w_comp * gaussian_pdf(domain, mu, var)
            comp_meta.append((mu, var, w_comp))

        mode_meta.append(
            {
                "mode": int(b),
                "mode_center": float(center),
                "mode_weight": float(w_mode[b]),
                "components": comp_meta,
            }
        )

    domain, density, _dx = _normalize_pdf(domain, density)

    # entropy-based effective number of active modes (1..3)
    eff_modes = float(np.exp(-np.sum(w_mode * np.log(np.maximum(w_mode, 1e-12)))))

    return density, {
        "x": x,
        "f": float(f),
        "sep": float(sep),
        "overlap_factor": float(overlap_factor),
        "eff_modes": eff_modes,
        "mode_centers": centers.tolist(),
        "mode_weights": w_mode.tolist(),
        "modes": mode_meta,
    }


def sample_from_density(domain: np.ndarray, density: np.ndarray, n_samples: int, rng: np.random.Generator) -> np.ndarray:
    """Inverse-CDF sampling on a discretized 1D density."""
    domain = np.asarray(domain, dtype=float)
    density = np.asarray(density, dtype=float)
    dx = np.abs(np.gradient(domain))
    mass = np.maximum(density, 0.0) * dx
    mass_sum = float(np.sum(mass))
    if mass_sum <= 0:
        return rng.choice(domain, size=int(n_samples), replace=True)

    cdf = np.cumsum(mass / mass_sum)
    cdf[-1] = 1.0
    u = rng.random(int(n_samples))
    return np.interp(u, cdf, domain)


def make_synthetic_dataset(
    n_stations: int,
    samples_per_station: int,
    *,
    x_min: float,
    x_max: float,
    domain: np.ndarray,
    seed: int,
):
    rng = np.random.default_rng(seed)
    x_all = np.linspace(float(x_min), float(x_max), int(n_stations), dtype=float).reshape(-1, 1)

    series_all: list[np.ndarray] = []
    y_data_true: list[tuple[np.ndarray, np.ndarray]] = []
    meta_all: list[dict] = []

    for i in range(int(n_stations)):
        x_i = float(x_all[i, 0])
        density_i, meta_i = build_station_density(x_i, np.asarray(domain, dtype=float))
        samples_i = sample_from_density(domain, density_i, int(samples_per_station), rng)

        y_data_true.append((np.asarray(domain, dtype=float), density_i))
        series_all.append(np.asarray(samples_i, dtype=float))
        meta_all.append(meta_i)

    return x_all, series_all, y_data_true, meta_all


In [ ]:

# Build synthetic stations
x_all, series_all, y_data_true, meta_all = make_synthetic_dataset(
    N_STATIONS,
    SAMPLES_PER_STATION,
    x_min=X_MIN,
    x_max=X_MAX,
    domain=Y_DOMAIN,
    seed=SEED,
)

# Train/test split by station id
n_train = int(TRAIN_RATIO * len(x_all))
perm = np.random.permutation(len(x_all))
train_idx = np.sort(perm[:n_train])
test_idx = np.sort(perm[n_train:])

x_train = x_all[train_idx]
x_test = x_all[test_idx]
train_series = [series_all[i] for i in train_idx]
test_series = [series_all[i] for i in test_idx]

# Fixed local indices used for side-by-side comparisons across K
compare_train_local = np.linspace(0, len(train_idx) - 1, min(N_COMPARISON_STATIONS, len(train_idx)), dtype=int)
compare_test_local = np.linspace(0, len(test_idx) - 1, min(N_COMPARISON_STATIONS, len(test_idx)), dtype=int)

logging.info(
    "Synthetic dataset ready: N=%d | train=%d | test=%d | samples/station=%d",
    len(x_all), len(train_idx), len(test_idx), SAMPLES_PER_STATION
)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.offsetbox import OffsetImage, AnnotationBbox, HPacker, VPacker, TextArea, AuxTransformBox, DrawingArea
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D

plt.rcdefaults()

# ---------- compute field ----------
x_vis = np.linspace(float(X_MIN), float(X_MAX), 1400)
true_vis = np.zeros((len(Y_DOMAIN), len(x_vis)), dtype=float)
for ix, xv in enumerate(x_vis):
    p, _ = build_station_density(float(xv), Y_DOMAIN)
    true_vis[:, ix] = p

log_true = np.log10(true_vis + 1e-12)

# ---------- pick points ----------
n_pts = 4
edges = np.linspace(float(X_MIN), float(X_MAX), n_pts + 1)
selected_x = 0.5 * (edges[:-1] + edges[1:])
selected_densities = [build_station_density(float(xv), Y_DOMAIN) for xv in selected_x]

# ---------- robust color scaling ----------
vals = log_true.ravel()
vmin = float(np.quantile(vals, 0.02))
vmax = float(np.quantile(vals, 0.998))

# ---------- colormap ----------
n = 512
t = np.linspace(0, 1, n)
colors = np.empty((n, 4))

c_white = np.array([1.0, 1.0, 1.0, 1.0])
c_light = np.array([0.78, 0.92, 0.90, 1.0])
c_mid   = np.array([0.25, 0.62, 0.65, 1.0])
c_deep  = np.array([0.12, 0.30, 0.50, 1.0])
c_dark  = np.array([0.08, 0.10, 0.28, 1.0])

anchors = [
    (0.00, c_white),
    (0.15, c_light),
    (0.40, c_mid),
    (0.72, c_deep),
    (1.00, c_dark),
]

for i, ti in enumerate(t):
    for k in range(len(anchors) - 1):
        t0, col0 = anchors[k]
        t1, col1 = anchors[k + 1]
        if ti <= t1 or k == len(anchors) - 2:
            a = np.clip((ti - t0) / (t1 - t0), 0, 1)
            a = a * a * (3 - 2 * a)
            colors[i] = (1 - a) * col0 + a * col1
            break

cmap = LinearSegmentedColormap.from_list("white_teal_indigo", colors, N=n)
norm = mcolors.Normalize(vmin=vmin, vmax=vmax, clip=True)

# ---------- styling ----------
ink = "#080808"
highlight = "#E8400E"

xspan = float(x_vis.max() - x_vis.min())
half_w = 0.065 * xspan
shape_pow = 0.55
trim_thresh = 0.0000001

# ---------- plot ----------
fig, ax = plt.subplots(figsize=(15.5, 5.2), facecolor="white")
ax.set_facecolor("white")

ax.imshow(
    log_true, origin="lower", aspect="auto",
    extent=[x_vis.min(), x_vis.max(), Y_DOMAIN.min(), Y_DOMAIN.max()],
    cmap=cmap, norm=norm, interpolation="bilinear",
)

f_vis = np.array([latent_mean(float(xv)) for xv in x_vis])
ax.plot(x_vis, f_vis, color="white", linewidth=6.5, alpha=0.55, zorder=5)
ax.plot(x_vis, f_vis, color=ink, linewidth=3.2, alpha=0.95, zorder=6)

for xv, (p, _meta) in zip(selected_x, selected_densities):
    yv = latent_mean(float(xv))
    p = np.asarray(p, float)
    p = np.clip(p, 0.0, None)
    p_norm = p / (p.max() + 1e-12)

    mask = p_norm >= trim_thresh
    if not np.any(mask):
        continue
    idx = np.where(mask)[0]
    i_lo, i_hi = idx[0], idx[-1]

    y_trim = Y_DOMAIN[i_lo:i_hi + 1]
    p_trim = p_norm[i_lo:i_hi + 1]
    w_trim = half_w * (p_trim ** shape_pow)

    ax.fill_betweenx(
        y_trim, xv, xv + w_trim,
        facecolor=highlight, alpha=0.38, edgecolor="none", zorder=8,
    )
    ax.plot(xv + w_trim, y_trim, color=highlight, linewidth=2.8, alpha=0.95, zorder=9)

ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values():
    s.set_visible(False)

# ---------- compact legend ----------
# line handles
h_fx = Line2D([0], [0], color=ink, linewidth=3.2, label=r"$f(x)$")
h_sample = Line2D([0], [0], color=highlight, linewidth=2.8, label=r"$p(y \mid x)$ at sample")

# gradient swatch as a small image handle
grad_w_px = 80
grad_arr = np.linspace(0, 1, grad_w_px).reshape(1, -1)
grad_rgba = cmap(np.repeat(grad_arr, 10, axis=0))  # thin strip

# DrawingArea to embed the gradient
da = DrawingArea(32, 8, 0, 0)
grad_im = plt.matplotlib.image.BboxImage(da.get_window_extent, cmap=cmap, norm=plt.Normalize(0, 1))
# Instead, use OffsetImage approach inside legend via a proxy handler

# --- use handler_map to render gradient in legend ---
from matplotlib.legend_handler import HandlerBase

class GradientHandler(HandlerBase):
    def __init__(self, cmap, width=50, height=8):
        self.cmap = cmap
        self.gw = width
        self.gh = height
        super().__init__()

    def create_artists(self, legend, orig_handle, xdescent, ydescent,
                       width, height, fontsize, trans):
        from matplotlib.image import BboxImage
        from matplotlib.transforms import Bbox, TransformedBbox

        bb = Bbox.from_bounds(xdescent, ydescent, width, height)
        tbb = TransformedBbox(bb, trans)
        img = BboxImage(tbb, cmap=self.cmap, norm=plt.Normalize(0, 1))
        img.set_data(np.linspace(0, 1, self.gw).reshape(1, -1))
        return [img]

# dummy handle for gradient
h_grad = Line2D([0], [0], color="none", label=r"$\log_{10}\,p(y \mid x)$")

leg = ax.legend(
    handles=[h_fx, h_sample, h_grad],
    loc="upper left",
    fontsize=15,
    frameon=True,
    fancybox=True,
    framealpha=0.93,
    edgecolor="#cccccc",
    borderpad=0.5,
    labelspacing=0.5,
    handlelength=2.2,
    handletextpad=0.6,
    handler_map={h_grad: GradientHandler(cmap)},
)
leg.set_zorder(15)

fig.suptitle(
    r"Selected points: same $f(x)$, very different local densities",
    y=0.99, fontsize=17, color=ink,
)
fig.subplots_adjust(left=0.01, right=0.99, bottom=0.02, top=0.90)
plt.show()


## GGMP run helper

For each `K` we do:

1. Fit station-level fixed-weight `K`-component GMMs on train-series samples
2. Train GGMP component GPs via MCMC
3. Optimize shared weights with EM on density objective
4. Evaluate held-out test stations + plot diagnostics


In [ ]:

def run_disjoint_experiment(K_run: int, *, run_name: str | None = None):
    from scipy.stats import norm as stats_norm
    K_run = int(K_run)
    run_name = run_name or f"K{K_run}_{time.strftime('%Y%m%d_%H%M%S')}"
    run_dir = OUT_DIR / run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    logging.info("=" * 70)
    logging.info("Running synthetic disjoint experiment with K=%d", K_run)
    logging.info("run_dir=%s", run_dir)

    # 1) Fit station-level GMM means/vars with fixed shared weights (uniform)
    w_init = np.ones(K_run, dtype=float) / float(K_run)
    means_mat = np.zeros((len(train_series), K_run), dtype=float)
    vars_mat = np.zeros((len(train_series), K_run), dtype=float)

    t0 = time.time()
    for i, y in enumerate(train_series, start=1):
        means_init = np.quantile(y, np.linspace(0.1, 0.9, K_run))
        m, v = fit_gmm_fixed_weights(
            y,
            K_run,
            w_init,
            means_init=means_init,
            max_iter=int(GMM_MAX_ITER),
            tol=float(GMM_TOL),
        )
        means_mat[i - 1] = m
        vars_mat[i - 1] = v

    logging.info("Station GMM fitting done in %.2fs", time.time() - t0)

    # 2) Build GGMP model
    y_data_train = [empirical_pdf_from_samples(y, bins=int(HIST_BINS)) for y in train_series]

    weights_bounds = np.array([[float(WEIGHT_FLOOR), 1.0]] * K_run, dtype=float)
    hps_list = []
    hps_bounds_list = []
    for k in range(K_run):
        mean_k = float(np.mean(means_mat[:, k]))
        # 1D input: [signal, lengthscale, mean]
        hps_list.append(np.array([1.0, 1.0, mean_k], dtype=float))
        hps_bounds_list.append(
            np.array(
                [
                    [0.1, 10.0],
                    [0.01, 6.0],
                    [mean_k - 40.0, mean_k + 40.0],
                ],
                dtype=float,
            )
        )

    hps_obj = hyperparameters(w_init.copy(), weights_bounds, hps_list, hps_bounds_list)
    gp_init_kwargs, _ = build_gp_init_kwargs(use_gpu=USE_GPU, gpu_engine=GPU_ENGINE)

    model = GGMP(
        x_train,
        y_data_train,
        hps_obj=hps_obj,
        likelihood_terms=K_run,
        gp_init_kwargs=gp_init_kwargs,
        gp_device_ids=_parse_device_ids(GP_DEVICE_IDS),
    )

    init_mean = [means_mat[:, k] for k in range(K_run)]
    init_std = [np.sqrt(vars_mat[:, k]) for k in range(K_run)]
    model.initLikelihoods(init_mean=init_mean, init_std=init_std, weights=w_init)
    model.initGPs()

    # 3) Train component GPs + EM optimize shared weights
    trained_hps = train_component_gps_mcmc(
        model,
        hps_obj,
        n_updates_gp=int(N_UPDATES_GP),
        mcmc_until_converged=bool(MCMC_UNTIL_CONVERGED),
        mcmc_chunk=int(MCMC_CHUNK),
        mcmc_max_total=int(MCMC_MAX_TOTAL),
        mcmc_tol_rel=float(MCMC_TOL_REL),
        mcmc_patience=int(MCMC_PATIENCE),
        gp_parallel=bool(GP_PARALLEL),
        gp_workers=(int(GP_WORKERS) if GP_WORKERS is not None else None),
        blas_threads_per_gp=(int(BLAS_THREADS_PER_GP) if BLAS_THREADS_PER_GP is not None else None),
        run_dir=run_dir,
        save_gp_mcmc=bool(SAVE_GP_MCMC),
        gp_mcmc_thin=int(GP_MCMC_THIN),
        save_gp_mcmc_chunks=bool(SAVE_GP_MCMC_CHUNKS),
    )

    terms, ll_comp = prepare_station_terms_density(model, trained_hps)
    w_em, w_hist, obj_hist = optimize_weights_em_density(
        terms,
        K=K_run,
        weight_floor=float(WEIGHT_FLOOR),
        max_iter=int(EM_MAX_ITER),
        tol_l1=float(EM_TOL_L1),
        log_every=int(EM_LOG_EVERY),
        w0=w_init,
    )

    logging.info("K=%d | final EM weights=%s", K_run, np.array2string(w_em, precision=5))

    # Update model weights
    model.hps_obj.set(w_em, trained_hps)
    for k in range(K_run):
        try:
            model.likelihoods[k].set_weight(float(w_em[k]))
        except Exception:
            pass

    # 4) Test metrics + predictions on common Y_DOMAIN
    y_data_test = [empirical_pdf_from_samples(y, bins=int(HIST_BINS)) for y in test_series]

    mu_test = np.empty((len(test_idx), K_run), dtype=float)
    var_gp_test = np.empty((len(test_idx), K_run), dtype=float)
    for k in range(K_run):
        gp = model.gps[k]
        model._safe_set_hyperparameters(gp, trained_hps[k])
        pm = gp.posterior_mean(x_test)
        pc = gp.posterior_covariance(x_test, variance_only=True)
        mu_test[:, k] = np.asarray(_get_key(pm, ("m(x)", "f(x)")), dtype=float).reshape(-1)
        var_gp_test[:, k] = np.maximum(
            np.asarray(_get_key(pc, ("v(x)", "v", "variance", "cov")), dtype=float).reshape(-1),
            0.0,
        )

    var_comp_global = np.array([float(np.mean(model.likelihoods[k].variance)) for k in range(K_run)], dtype=float)
    var_comp_global = np.maximum(var_comp_global, 1e-9)

    metrics_emp = []
    pred_test_grid = np.zeros((len(test_idx), len(Y_DOMAIN)), dtype=float)
    true_test_grid = np.vstack([y_data_true[int(sid)][1] for sid in test_idx])

    for j, (domain, density) in enumerate(y_data_test):
        # Empirical-histogram-based metric
        domain, p_obs, dx = _normalize_pdf(domain, density)
        mix = np.zeros_like(domain, dtype=float)
        for k in range(K_run):
            sigma = float(np.sqrt(max(var_gp_test[j, k] + var_comp_global[k], 1e-12)))
            mix += float(w_em[k]) * stats_norm.pdf(domain, loc=float(mu_test[j, k]), scale=sigma)
        mix = np.maximum(mix, 1e-300)
        mix = mix / float(np.sum(mix * dx) + 1e-300)

        bdist = bhattacharyya_distance(domain, p_obs, mix)
        kl_pq = kl_divergence(domain, p_obs, mix)
        kl_qp = kl_divergence(domain, mix, p_obs)
        w1 = wasserstein_1d(domain, p_obs, mix)
        l1 = float(np.sum(np.abs(p_obs - mix) * dx))
        metrics_emp.append(
            {
                "station_id": int(test_idx[j]),
                "bhattacharyya": float(bdist),
                "kl_sym": float(0.5 * (kl_pq + kl_qp)),
                "wasserstein1": float(w1),
                "l1": float(l1),
            }
        )

        # Prediction on common Y_DOMAIN for direct K-to-K visual comparison
        mix_common = np.zeros_like(Y_DOMAIN, dtype=float)
        for k in range(K_run):
            sigma = float(np.sqrt(max(var_gp_test[j, k] + var_comp_global[k], 1e-12)))
            mix_common += float(w_em[k]) * stats_norm.pdf(Y_DOMAIN, loc=float(mu_test[j, k]), scale=sigma)
        Yn, mix_common, _ = _normalize_pdf(Y_DOMAIN, mix_common)
        pred_test_grid[j] = mix_common

    # True-density-based metrics on common grid
    metrics_true = []
    for j in range(len(test_idx)):
        p = true_test_grid[j]
        q = pred_test_grid[j]
        bdist = bhattacharyya_distance(Y_DOMAIN, p, q)
        kl_pq = kl_divergence(Y_DOMAIN, p, q)
        kl_qp = kl_divergence(Y_DOMAIN, q, p)
        w1 = wasserstein_1d(Y_DOMAIN, p, q)
        dx = np.abs(np.gradient(Y_DOMAIN))
        l1 = float(np.sum(np.abs(p - q) * dx))
        metrics_true.append(
            {
                "station_id": int(test_idx[j]),
                "bhattacharyya": float(bdist),
                "kl_sym": float(0.5 * (kl_pq + kl_qp)),
                "wasserstein1": float(w1),
                "l1": float(l1),
            }
        )

    # Predictive density field over x-grid
    x_grid = np.linspace(float(np.min(x_all)), float(np.max(x_all)), 240).reshape(-1, 1)
    y_grid = Y_DOMAIN.copy()

    mu_grid = np.empty((x_grid.shape[0], K_run), dtype=float)
    var_gp_grid = np.empty((x_grid.shape[0], K_run), dtype=float)
    for k in range(K_run):
        gp = model.gps[k]
        model._safe_set_hyperparameters(gp, trained_hps[k])
        pm = gp.posterior_mean(x_grid)
        pc = gp.posterior_covariance(x_grid, variance_only=True)
        mu_grid[:, k] = np.asarray(_get_key(pm, ("m(x)", "f(x)")), dtype=float).reshape(-1)
        var_gp_grid[:, k] = np.maximum(np.asarray(_get_key(pc, ("v(x)", "v", "variance", "cov")), dtype=float).reshape(-1), 0.0)

    pdf_grid = np.zeros((y_grid.size, x_grid.shape[0]), dtype=float)
    dy = np.abs(np.gradient(y_grid))
    for ix in range(x_grid.shape[0]):
        mix = np.zeros_like(y_grid)
        for k in range(K_run):
            sigma = float(np.sqrt(max(var_gp_grid[ix, k] + var_comp_global[k], 1e-12)))
            mix += float(w_em[k]) * stats_norm.pdf(y_grid, loc=float(mu_grid[ix, k]), scale=sigma)
        mix = np.maximum(mix, 1e-300)
        mix = mix / float(np.sum(mix * dy) + 1e-300)
        pdf_grid[:, ix] = mix

    # -----------------------
    # Plots for this K
    # -----------------------

    # A) Train diagnostics on fixed stations (true density vs fitted station mixture)
    n_show = len(compare_train_local)
    fig, axes = plt.subplots(1, n_show, figsize=(4.4 * n_show, 3.6), squeeze=False)
    axes = axes.flatten()

    for ax, i_local in zip(axes, compare_train_local):
        sid = int(train_idx[i_local])
        means = means_mat[i_local]
        vars_ = vars_mat[i_local]

        true_density = y_data_true[sid][1]
        ax.plot(Y_DOMAIN, true_density, color="black", linewidth=2.2, label="true")

        mix = np.zeros_like(Y_DOMAIN)
        for k in range(K_run):
            comp_pdf = float(w_em[k]) * gaussian_pdf(Y_DOMAIN, float(means[k]), float(vars_[k]))
            mix += comp_pdf
            ax.plot(Y_DOMAIN, comp_pdf, linestyle="--", linewidth=1.0, alpha=0.7)
        ax.plot(Y_DOMAIN, mix, color="#d62828", linewidth=2.0, label="fitted mixture")
        ax.set_title(f"train station {sid}")
        ax.set_xlabel("y")

    axes[0].set_ylabel("density")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=True)
    fig.suptitle(f"Train density fit diagnostics (K={K_run})", y=1.05)
    fig.tight_layout()
    plt.show()

    # B) EM traces
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.6))
    ax1.plot(obj_hist, color="black", linewidth=2)
    ax1.set_title(f"EM objective (K={K_run})")
    ax1.set_xlabel("iteration")
    ax1.set_ylabel("objective")

    for k in range(K_run):
        ax2.plot(w_hist[:, k], linewidth=1.8, label=f"w[{k}]")
    ax2.set_title("EM weights")
    ax2.set_xlabel("iteration")
    ax2.set_ylabel("weight")
    if K_run <= 8:
        ax2.legend(frameon=True, fontsize=8)
    fig.tight_layout()
    plt.show()

    # C) Predictive density field
    Xg, Yg = np.meshgrid(x_grid[:, 0], y_grid)
    log_pdf = np.log10(pdf_grid + 1e-12)

    plt.figure(figsize=(13, 3.8))
    pcm = plt.pcolormesh(Xg, Yg, log_pdf, shading="auto", cmap="magma")
    cb = plt.colorbar(pcm)
    cb.set_label("log10 predictive density")
    plt.title(f"Predictive density field (K={K_run})")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.tight_layout()
    plt.show()

    # D) Fixed held-out station overlays (true vs predicted)
    n_show_test = len(compare_test_local)
    fig, axes = plt.subplots(1, n_show_test, figsize=(4.4 * n_show_test, 3.6), squeeze=False)
    axes = axes.flatten()

    # precompute per-station true-based metric for title annotation
    metric_by_sid = {int(m["station_id"]): m for m in metrics_true}

    for ax, j_local in zip(axes, compare_test_local):
        sid = int(test_idx[j_local])
        p_true = true_test_grid[j_local]
        p_pred = pred_test_grid[j_local]
        m = metric_by_sid[sid]

        ax.plot(Y_DOMAIN, p_true, color="black", linewidth=2.2, label="true")
        ax.plot(Y_DOMAIN, p_pred, color="#d62828", linewidth=2.0, label="pred")
        ax.fill_between(Y_DOMAIN, 0.0, np.minimum(p_true, p_pred), color="#74c69d", alpha=0.35)
        ax.set_title(f"test station {sid}\nKLsym={m['kl_sym']:.3f}, W1={m['wasserstein1']:.3f}")
        ax.set_xlabel("y")

    axes[0].set_ylabel("density")
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=True)
    fig.suptitle(f"Held-out station reconstruction (K={K_run})", y=1.06)
    fig.tight_layout()
    plt.show()

    # Metric summaries
    def _summ(metric_list):
        out = {}
        for key in ("bhattacharyya", "kl_sym", "wasserstein1", "l1"):
            arr = np.asarray([m[key] for m in metric_list], dtype=float)
            out[key] = {
                "mean": float(np.mean(arr)),
                "median": float(np.median(arr)),
                "std": float(np.std(arr)),
            }
        return out

    summary_emp = _summ(metrics_emp)
    summary_true = _summ(metrics_true)

    for key in ("bhattacharyya", "kl_sym", "wasserstein1", "l1"):
        s = summary_true[key]
        logging.info(
            "K=%d | TRUE %s: mean=%.4f | median=%.4f | std=%.4f",
            K_run, key, s["mean"], s["median"], s["std"]
        )

    return {
        "K": K_run,
        "run_dir": run_dir,
        "w_em": w_em,
        "w_hist": w_hist,
        "obj_hist": obj_hist,
        "trained_hps": trained_hps,
        "metrics_emp": metrics_emp,
        "metrics_true": metrics_true,
        "summary_emp": summary_emp,
        "summary_true": summary_true,
        "means_mat": means_mat,
        "vars_mat": vars_mat,
        "x_grid": x_grid,
        "y_grid": y_grid,
        "pdf_grid": pdf_grid,
        "true_test_grid": true_test_grid,
        "pred_test_grid": pred_test_grid,
    }


## Train with K

In [ ]:

# K = 3
res_k3 = run_disjoint_experiment(3, run_name=f"K3_{time.strftime('%Y%m%d_%H%M%S')}")